# K-Means clustering on combined_cleaned.csv

This notebook loads `combined_cleaned.csv`, prepares features, runs **K-Means clustering**, visualizes clusters (PCA 2D), and saves `combined_cleaned_kmeans.csv`.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

%matplotlib inline


In [ ]:
data = pd.read_csv('combined_cleaned.csv')
print('Data shape:', data.shape)
display(data.head())


In [ ]:
if 'heart_disease' in data.columns:
    X = data.drop(columns=['heart_disease'])
else:
    X = data.copy()

X = pd.get_dummies(X, drop_first=True)
print('Encoded features shape:', X.shape)


In [ ]:
imputer = SimpleImputer(strategy='median')
X_imputed = imputer.fit_transform(X)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_imputed)

print('Clustering matrix shape:', X_scaled.shape)


In [ ]:
k_values = range(2, 11)
inertias = []
sil_scores = []

silhouette_sample_size = min(10000, X_scaled.shape[0])

for k in k_values:
    km = KMeans(n_clusters=k, random_state=42, n_init='auto')
    labels = km.fit_predict(X_scaled)
    inertias.append(km.inertia_)
    sil_scores.append(
        silhouette_score(
            X_scaled,
            labels,
            sample_size=silhouette_sample_size,
            random_state=42,
        )
    )

best_k = int(list(k_values)[int(np.argmax(sil_scores))])
print('Best k by silhouette:', best_k)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(list(k_values), inertias, marker='o')
axes[0].set_title('Elbow method (inertia)')
axes[0].set_xlabel('k')
axes[0].set_ylabel('Inertia')
axes[0].grid(True, alpha=0.3)

axes[1].plot(list(k_values), sil_scores, marker='o')
axes[1].set_title(f'Silhouette score (sample={silhouette_sample_size})')
axes[1].set_xlabel('k')
axes[1].set_ylabel('Silhouette')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


In [ ]:
kmeans = KMeans(n_clusters=best_k, random_state=42, n_init='auto')
clusters = kmeans.fit_predict(X_scaled)

out = data.copy()
out['cluster'] = clusters

print('Cluster counts:')
print(out['cluster'].value_counts().sort_index())

pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)

plt.figure(figsize=(7, 5))
plt.scatter(X_pca[:, 0], X_pca[:, 1], c=clusters, s=5, cmap='tab10', alpha=0.6)
plt.title(f'K-Means clusters (k={best_k}) visualized with PCA')
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.tight_layout()
plt.show()

out_path = 'combined_cleaned_kmeans.csv'
out.to_csv(out_path, index=False)
print('Saved:', out_path)
